# RawFileReader Python Adapter — Demo

This notebook shows how to read Thermo Fisher `.raw` mass-spectrometry files
in Google Colab using the **rawfilereader** Python package.

**Steps**
1. Install .NET 8 and pythonnet (cells 1–2)
2. Clone the repo and set up the package (cell 3)
3. Define a reload helper (cell 4)
4. Download the RawFileReader DLLs (cell 5)
5. Set the `.raw` file path (cell 6 — uses `sample.raw` bundled in the repo)
6. Explore the file with the adapter API (cells 7+)

> **After a `git pull`** run `reload_rawfilereader()` in any cell to pick up
> the latest code without restarting the kernel.

## Step 1 — Install .NET 8 Runtime

In [ ]:
%%bash
set -e

# Register the Microsoft package feed
wget -q https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/packages-microsoft-prod.deb \
     -O packages-microsoft-prod.deb
dpkg -i packages-microsoft-prod.deb > /dev/null
rm packages-microsoft-prod.deb

# Install .NET 8 runtime
apt-get update -qq
apt-get install -y dotnet-runtime-8.0 2>&1 | grep -E '(Setting up|already|error)' || true

echo
echo "--- Installed runtimes ---"
dotnet --list-runtimes

## Step 2 — Install pythonnet

pythonnet must be installed **after** .NET so it links against the correct runtime.

In [ ]:
!pip install -q --force-reinstall pythonnet

# pythonnet defaults to Mono on Linux — force CoreCLR (.NET 8)
import pythonnet
pythonnet.load("coreclr")
import clr
print("pythonnet OK — CLR version:", clr.__version__)

## Step 3 — Clone the repository and install the package

In [ ]:
import os, sys

if not os.path.isdir("/content/RawFileReaderPyAdapter"):
    !git clone -q https://github.com/mzzzhunter/RawFileReaderPyAdapter.git /content/RawFileReaderPyAdapter
else:
    !git -C /content/RawFileReaderPyAdapter pull -q

# Add to sys.path so the current kernel finds the package immediately
# (pip install -e does not update sys.path in an already-running Colab kernel)
repo_path = "/content/RawFileReaderPyAdapter"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import rawfilereader
print("rawfilereader", rawfilereader.__version__, "ready.")

In [ ]:
import importlib, sys

def reload_rawfilereader():
    """Re-import rawfilereader from disk (call after git pull)."""
    # Skip rawfilereader.loader — reloading it resets _loaded=False which
    # triggers slow .NET assembly re-initialization on the next open() call.
    mods = sorted(
        [k for k in sys.modules
         if k.startswith("rawfilereader") and k != "rawfilereader.loader"],
        reverse=True,   # submodules before parent
    )
    for name in mods:
        importlib.reload(sys.modules[name])
    print("rawfilereader reloaded.")

## Step 4 — Download the RawFileReader DLLs

In [ ]:
!python /content/RawFileReaderPyAdapter/download_dlls.py \
    --libs-dir /content/RawFileReaderPyAdapter/libs \
    --no-env

import os
os.environ["RAWFILEREADER_LIBS"] = "/content/RawFileReaderPyAdapter/libs"
print("RAWFILEREADER_LIBS:", os.environ["RAWFILEREADER_LIBS"])

## Step 5 — Set the `.raw` file path

`sample.raw` ships with the repository and is already present after the clone
in Step 3. `RAW_FILE` points to it automatically.

To use a different file, change `RAW_FILE` to any path accessible in the
Colab runtime (e.g. a file uploaded via the Files panel or mounted from Drive).

In [ ]:
import os

RAW_FILE = "/content/RawFileReaderPyAdapter/sample.raw"

if not os.path.isfile(RAW_FILE):
    raise FileNotFoundError(f"sample.raw not found at {RAW_FILE} — did the clone in Step 3 complete?")

print(f"Using file: {RAW_FILE}  ({os.path.getsize(RAW_FILE) / 1e6:.1f} MB)")

## Step 6 — File summary

In [ ]:
from rawfilereader import RawFileAdapter

with RawFileAdapter(RAW_FILE) as rf:
    fi = rf.get_file_info()
    si = rf.get_sample_info()
    first, last = rf.get_scan_range()

    print(f"File       : {fi.file_name}")
    print(f"Date       : {fi.creation_date}")
    print(f"Operator   : {fi.operator}")
    print(f"Sample     : {si.sample_name}")
    print(f"Vial       : {si.vial}")
    print(f"Instrument : {fi.instrument_name}  ({fi.instrument_serial_number})")
    print(f"Scans      : {first} – {last}")
    print(f"RT range   : {rf.get_start_time():.2f} – {rf.get_end_time():.2f} min")

## Step 7 — Scan filters present in the file

In [ ]:
with RawFileAdapter(RAW_FILE) as rf:
    filters = rf.get_filters()

print(f"{len(filters)} unique filter(s):")
for f in filters:
    print(" ", f)

## Step 8 — Total Ion Chromatogram (TIC)

In [ ]:
import matplotlib.pyplot as plt

with RawFileAdapter(RAW_FILE) as rf:
    tic = rf.get_chromatogram(trace_type="TIC")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tic.times, tic.intensities, linewidth=0.8)
ax.set_xlabel("Retention time (min)")
ax.set_ylabel("Intensity")
ax.set_title("Total Ion Chromatogram")
plt.tight_layout()
plt.show()

## Step 9 — Inspect a single scan

Change `SCAN` to any scan number in the file.

In [ ]:
SCAN = 1   # ← change this

with RawFileAdapter(RAW_FILE) as rf:
    info    = rf.get_scan_info(SCAN)
    centroid = rf.get_centroid_stream(SCAN)

print(f"Scan        : {info.scan_number}")
print(f"MS order    : {info.ms_order}")
print(f"RT          : {info.retention_time:.4f} min")
print(f"Filter      : {info.scan_filter}")
print(f"Inj. time   : {info.injection_time:.2f} ms")
if info.ms_order >= 2:
    print(f"Precursor   : {info.precursor_mass:.4f}  z={info.precursor_charge}")
    print(f"CE          : {info.collision_energy:.1f} eV")
print(f"Peaks       : {len(centroid.masses)}")

# Plot the spectrum
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(centroid.masses, 0, centroid.intensities, linewidth=0.6)
ax.set_xlabel("m/z")
ax.set_ylabel("Intensity")
ax.set_title(f"Scan {SCAN}  RT={info.retention_time:.3f} min  [{info.scan_filter}]")
plt.tight_layout()
plt.show()

## Step 10 — Iterate all MS1 scans and collect top peaks

In [ ]:
records = []

with RawFileAdapter(RAW_FILE) as rf:
    first, last = rf.get_scan_range()
    for scan in range(first, last + 1):
        info = rf.get_scan_info(scan)
        if info.ms_order != 1:
            continue
        cd = rf.get_centroid_stream(scan)
        if cd.peaks:
            top_mass, top_int = max(cd.peaks, key=lambda p: p[1])
        else:
            top_mass, top_int = None, None
        records.append({
            "scan":     scan,
            "rt":       round(info.retention_time, 4),
            "peaks":    len(cd.masses),
            "top_mz":   round(top_mass, 4) if top_mass else None,
            "top_int":  round(top_int)      if top_int  else None,
        })

print(f"Collected {len(records)} MS1 scans.  First 5:")
for r in records[:5]:
    print(f"  scan={r['scan']}  RT={r['rt']}  peaks={r['peaks']}  "
          f"top_mz={r['top_mz']}  top_int={r['top_int']}")

## Step 11 — Export scan table to CSV

In [ ]:
import csv

out_path = "/content/ms1_scans.csv"
with open(out_path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} rows to {out_path}")

# Download the CSV to your computer
from google.colab import files
files.download(out_path)

## Step 12 — Extracted Ion Chromatogram (EIC) with scan-filter selection

In [ ]:
import matplotlib.pyplot as plt

with RawFileAdapter(RAW_FILE) as rf:
    # 1. List all scan filters present in the file
    filters = rf.get_filters()
    print(f"Available filters ({len(filters)}):")
    for i, f in enumerate(filters):
        print(f"  [{i}] {f}")

    # 2. Auto-select the first MS1 full-scan filter
    ms1_filter = next(
        (f for f in filters if "full ms" in f.lower()),
        filters[0],
    )
    print(f"\nSelected: {ms1_filter!r}")

    # 3. EIC for m/z 500–510 restricted to that filter
    eic = rf.get_chromatogram(
        trace_type="MassRange",
        mass_range="500.0-510.0",
        filter_string=ms1_filter,
    )

print(f"EIC points: {len(eic.times)}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(eic.times, eic.intensities, linewidth=0.8, color="steelblue")
ax.set_xlabel("Retention time (min)")
ax.set_ylabel("Intensity")
ax.set_title(f"EIC  m/z 500–510  |  {ms1_filter}")
plt.tight_layout()
plt.show()

## Step 13 — Full function coverage test

Calls every public method on `RawFileAdapter` and `RawFileThreadManager`
against `sample.raw`.  Each result is printed; failures show the exception
type and message instead of stopping the notebook.

In [ ]:
from rawfilereader import RawFileAdapter, RawFileThreadManager

SEP = "-" * 64

def _show(label, call):
    """Call `call()`, print a one-line summary, and return the value."""
    try:
        val = call()
        s = repr(val)
        preview = s[:120] + ("..." if len(s) > 120 else "")
        print(f"  [OK]   {label}: {preview}")
        return val
    except Exception as exc:
        print(f"  [FAIL] {label}: {type(exc).__name__}: {exc}")
        return None

# ── Pre-flight ────────────────────────────────────────────────────────────────
with RawFileAdapter(RAW_FILE) as rf:
    first, last = rf.get_scan_range()
    start_rt    = rf.get_start_time()
    end_rt      = rf.get_end_time()
    mid_rt      = (start_rt + end_rt) / 2

# ── File & Run Headers ────────────────────────────────────────────────────────
print(SEP); print("File & Run Headers"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    _show("get_file_info",                      rf.get_file_info)
    _show("get_scan_range",                     rf.get_scan_range)
    _show("get_start_time",                     rf.get_start_time)
    _show("get_end_time",                       rf.get_end_time)
    _show("get_instrument_count",               rf.get_instrument_count)
    _show("get_instrument_count_of_type('MS')", lambda: rf.get_instrument_count_of_type("MS"))
    _show("get_instrument_type(0)",             lambda: rf.get_instrument_type(0))
    _show("get_instrument_method_count",        rf.get_instrument_method_count)
    _show("get_instrument_method(0)",           lambda: rf.get_instrument_method(0))
    _show("get_all_instrument_names_from_method", rf.get_all_instrument_names_from_method)
    _show("get_instrument_data",                rf.get_instrument_data)

# ── Sample & Autosampler ──────────────────────────────────────────────────────
print(SEP); print("Sample & Autosampler"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    _show("get_sample_info",      rf.get_sample_info)
    _show("get_autosampler_info", rf.get_autosampler_info)

# ── Scan Filters ──────────────────────────────────────────────────────────────
print(SEP); print("Scan Filters"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    filters = _show("get_filters",      rf.get_filters)
    _show("get_auto_filters",           rf.get_auto_filters)
    filt0 = (filters or [""])[0]
    _show(f"get_filter_for_scan({first})",   lambda: rf.get_filter_for_scan(first))
    _show("get_filter_from_string",          lambda: rf.get_filter_from_string(filt0))
    _show("get_filtered_scan_numbers",       lambda: rf.get_filtered_scan_numbers(filt0)[:5])
    _show("get_filtered_scan_numbers_over_time",
          lambda: rf.get_filtered_scan_numbers_over_time(filt0, start_rt, mid_rt)[:5])
    _show(f"test_scan({first}, filter)",    lambda: rf.test_scan(first, filt0))

# ── Scan Data ─────────────────────────────────────────────────────────────────
print(SEP); print("Scan Data"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    _show(f"get_retention_time({first})",        lambda: rf.get_retention_time(first))
    _show("scan_number_from_retention_time",     lambda: rf.scan_number_from_retention_time(start_rt + 0.01))
    _show(f"get_scan_stats({first})",            lambda: rf.get_scan_stats(first))
    _show(f"get_centroid_stream({first})",       lambda: rf.get_centroid_stream(first))
    _show(f"get_profile_data({first})",          lambda: rf.get_profile_data(first))
    _show(f"get_scan_info({first})",             lambda: rf.get_scan_info(first))
    _show(f"get_scan_event_for_scan({first})",   lambda: rf.get_scan_event_for_scan(first))
    _show(f"get_scan_event_string({first})",     lambda: rf.get_scan_event_string(first))
    _show(f"get_scan_dependents({first})",       lambda: rf.get_scan_dependents(first))
    _show(f"get_mass_precision({first})",        lambda: rf.get_mass_precision(first))

# ── Chromatograms ─────────────────────────────────────────────────────────────
print(SEP); print("Chromatograms"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    filt0 = (rf.get_filters() or [""])[0]
    _show("get_chromatogram(TIC)",
          lambda: rf.get_chromatogram(trace_type="TIC"))
    _show("get_chromatogram(BasePeak)",
          lambda: rf.get_chromatogram(trace_type="BasePeak"))
    _show("get_chromatogram(MassRange 500-510)",
          lambda: rf.get_chromatogram(trace_type="MassRange", mass_range="500.0-510.0"))
    _show("get_chromatogram(TIC + filter_string)",
          lambda: rf.get_chromatogram(trace_type="TIC", filter_string=filt0))
    _show("get_chromatogram_by_time(TIC, start→mid)",
          lambda: rf.get_chromatogram_by_time(start_time=start_rt, end_time=mid_rt,
                                               trace_type="TIC"))
    _show("get_chromatogram_ex(MassRange 500-510)",
          lambda: rf.get_chromatogram_ex(trace_type="MassRange",
                                          mass_ranges=[(500.0, 510.0)]))

# ── Logs & Diagnostics ───────────────────────────────────────────────────────
print(SEP); print("Logs & Diagnostics"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    _show(f"get_trailer_data({first})",          lambda: rf.get_trailer_data(first))
    _show("get_trailer_header_info[:5]",         lambda: rf.get_trailer_header_info()[:5])
    _show("get_status_log_header_info[:5]",      lambda: rf.get_status_log_header_info()[:5])
    _show("get_status_log_for_scan",             lambda: rf.get_status_log_for_scan(first))
    _show("get_status_log_for_retention_time",   lambda: rf.get_status_log_for_retention_time(start_rt))
    _show("get_status_log_entries_count",        rf.get_status_log_entries_count)
    _show("get_status_log_at_position(0)",       lambda: rf.get_status_log_at_position(0))
    _show("get_tune_data_count",                 rf.get_tune_data_count)
    _show("get_tune_data(0)",                    lambda: rf.get_tune_data(0))
    n_err = _show("get_error_log_count",         rf.get_error_log_count)
    if n_err:
        _show("get_error_log_entry(0)",          lambda: rf.get_error_log_entry(0))
    else:
        print("  [--]   get_error_log_entry: skipped (error log is empty)")

# ── Averaging & Background Subtraction ───────────────────────────────────────
print(SEP); print("Averaging & Background Subtraction"); print(SEP)
with RawFileAdapter(RAW_FILE) as rf:
    _show("average_scans_in_range(first, first+4)",
          lambda: rf.average_scans_in_range(first, first + 4))
    _show("average_scans([first, first+1, first+2])",
          lambda: rf.average_scans([first, first + 1, first + 2]))
    _show("average_scans_in_time_range(start_rt, start_rt+0.5)",
          lambda: rf.average_scans_in_time_range(start_rt, start_rt + 0.5))
    _show("subtract_background(first, first+1)",
          lambda: rf.subtract_background(first, first + 1))

# ── RawFileThreadManager ─────────────────────────────────────────────────────
print(SEP); print("RawFileThreadManager"); print(SEP)
def _thread_mgr_test():
    with RawFileThreadManager(RAW_FILE) as mgr:
        acc = mgr.create_accessor()
        result = acc.get_scan_info(first)
        acc.close()
        return result

_show("create_accessor → get_scan_info", _thread_mgr_test)

print(SEP)
print("Coverage test complete.")